In [ ]:
RUN_NAME = "localiser_run4b_vgg16enc_whole"
ENCODER = "vgg16"          # "vgg16" -> localise.build_unet_pretrained; None -> scratch build_unet (run-3 arch)
WHOLE_IMAGE = True         # run 4b: whole 1024x576 images at batch 4 (config.LOC_BATCH_SIZE)
EPOCHS = 40
SEED = 28
LR_ENCODER, LR_DECODER, WARMUP_EPOCHS = 3e-5, 3e-4, 3
RUN4_WEIGHTS = "/kaggle/input/datasets/tataruteodor/localiser-run4-vgg16enc-patch/best.weights.h5"   # <- adjust to the attached dataset path

MERGED_STORE = "/kaggle/input/datasets/tataruteodor/ddsm-localiser-tensors-merged"   # <- adjust to the attached dataset path
LESION_STORE = "/kaggle/input/datasets/tataruteodor/localiser-tensors"          # <- adjust
SRC_DATASET  = "/kaggle/input/datasets/tataruteodor/src-l4b"                        # <- adjust (contains src/)
OUT = f"/kaggle/working/{RUN_NAME}"

In [ ]:
import os, shutil, sys, json
import tensorflow as tf

if os.path.exists("/kaggle/working/src"):
    shutil.rmtree("/kaggle/working/src")
shutil.copytree(os.path.join(SRC_DATASET, "src"), "/kaggle/working/src")
os.chdir("/kaggle/working")
sys.path.insert(0, "/kaggle/working")

print("TF", tf.__version__, "| GPUs", tf.config.list_physical_devices("GPU"))
tf.keras.mixed_precision.set_global_policy("float32")
assert tf.keras.mixed_precision.global_policy().name == "float32", "run 4 is float32 by design"

from src import config, localise
m = localise.build_unet_pretrained(ENCODER) if ENCODER else localise.build_unet()
print(f"{m.name}: {m.count_params():,} params")
del m
for d in (MERGED_STORE, LESION_STORE):
    print(d, sorted(os.listdir(d)))
assert os.path.exists(RUN4_WEIGHTS), RUN4_WEIGHTS
print("init weights:", RUN4_WEIGHTS, os.path.getsize(RUN4_WEIGHTS) // 2**20, "MB")

In [ ]:
from src.train_localiser_l4 import main
manifest = main(MERGED_STORE, LESION_STORE, OUT,
                encoder=ENCODER or "none", whole_image=WHOLE_IMAGE,
                epochs=EPOCHS, run_name=RUN_NAME, seed=SEED,
                init_weights=RUN4_WEIGHTS, lr_encoder=LR_ENCODER, lr_decoder=LR_DECODER,
                warmup_epochs=WARMUP_EPOCHS)
print(json.dumps({k: manifest[k] for k in (
    "run", "regime", "params", "steps_per_epoch", "epochs_run", "first_nan_epoch",
    "best_epoch", "val_hard_iou_at_best_epoch", "val_dice_at_best_epoch",
    "val_loss_at_best_epoch", "best_valloss_epoch", "max_val_hard_iou_any_epoch",
    "wall_time_s") if k in manifest}, indent=2))

In [ ]:
# Same scorer as the local gate (localise_eval), pointed at the per-lesion
# store; comparable with run 1's val box_iou_inclusive_mean 0.3853.
from pathlib import Path
from src import localise_eval
localise_eval.TENSORS = Path(LESION_STORE)
model = localise_eval._load_model(f"{OUT}/best.weights.h5", config.LOC_BASE_FILTERS,
                                  config.LOC_DEPTH, encoder=ENCODER or "none")
df = localise_eval.evaluate_split(model, "val")
rep = localise_eval.report(df, "val")
df.to_csv(f"{OUT}/localiser_boxes_val.csv", index=False)
json.dump([rep], open(f"{OUT}/localiser_metrics_val.json", "w"), indent=2)
print({k: round(rep[k], 4) for k in ("box_iou_inclusive_mean", "detection_rate",
                                     "box_iou_mean_given_detected", "detect_at_iou50", "dice_mean")})

In [ ]:
# Download convention on the laptop:
#   outputs/weights/localiser_run4b_vgg16enc_whole/{best.weights.h5, best_valloss.weights.h5, manifest.json, history.csv}
print(sorted(os.listdir(OUT)))
shutil.make_archive(f"/kaggle/working/{RUN_NAME}", "zip", OUT)
print(f"-> /kaggle/working/{RUN_NAME}.zip")
